# Wikipedia Chat Bot Demo
This notebook builds a chat system around a Wikipedia article. It retrievs page content, preparing it for retrieval, and then answering user questions using that source text.

The goal is to show how a context-aware bot can be created in a Jupyter notebook without relying on external LLM services.

In [15]:
import requests
import re
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Imports ready and environment is configured for text retrieval.")

Imports ready and environment is configured for text retrieval.


## Fetch Wikipedia Content

The next step is to retrieve a Wikipedia page using the public API. I chose the article for **Ada Lovelace**, which is a strong example of a historical computing figure and is not the same page used in the lecture video.

In [16]:
import requests  # Make sure to import the requests library

WIKIPEDIA_TITLE = "Ada Lovelace"
API_URL = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "prop": "extracts",
    "titles": WIKIPEDIA_TITLE,
    "format": "json",
    "explaintext": 1,
    "redirects": 1,
}

# Add a User-Agent header to identify your application
headers = {
    "User-Agent": "YourAppName/1.0 (your.email@example.com) Python/requests"
}

# Include the headers in the request
response = requests.get(API_URL, params=params, headers=headers, timeout=20)
response.raise_for_status()
data = response.json()
page = next(iter(data["query"]["pages"].values()))
raw_text = page.get("extract", "")

print(f"Fetched page: {page.get('title', 'Unknown')} ({len(raw_text)} characters)")
print(raw_text[:1000])

Fetched page: Ada Lovelace (40605 characters)
Augusta Ada King, Countess of Lovelace (née Byron; 10 December 1815 – 27 November 1852), also known as Ada Lovelace, was an English mathematician and writer chiefly known for work on Charles Babbage's proposed mechanical general-purpose computer, the analytical engine. She was the first to recognise the machine had applications beyond pure calculation. Lovelace is often considered the first computer programmer.
Lovelace was the only legitimate child of poet Lord Byron and reformer Anne Isabella Milbanke. Lord Byron separated from his wife a month after Ada was born, and died when she was eight. Although often ill in childhood, Lovelace pursued her studies assiduously. She married William King in 1835. King was a Baron, and was created Viscount Ockham and 1st Earl of Lovelace in 1838. The name Lovelace was chosen because Ada was descended from the extinct Baron Lovelaces. The title given to her husband thus made Ada the Countess of Lovelace.

## Prepare Text for Chat Interface

A retrieval-based chat bot needs the source text to be split into manageable pieces. I convert the Wikipedia extract into chunks and build a TF-IDF matrix so the bot can match questions to the most relevant passages.

In [17]:
def split_into_chunks(text, max_chunk_size=800):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_chunk_size:
            current = current + ("\n\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            if len(para) <= max_chunk_size:
                current = para
            else:
                for i in range(0, len(para), max_chunk_size):
                    chunks.append(para[i : i + max_chunk_size])
                current = ""
    if current:
        chunks.append(current)
    return chunks

chunks = split_into_chunks(raw_text)
vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(chunks)

print(f"Created {len(chunks)} text chunks for retrieval.")
print("Sample chunk:\n", chunks[0][:400].replace('\n', ' '))

Created 71 text chunks for retrieval.
Sample chunk:
 Augusta Ada King, Countess of Lovelace (née Byron; 10 December 1815 – 27 November 1852), also known as Ada Lovelace, was an English mathematician and writer chiefly known for work on Charles Babbage's proposed mechanical general-purpose computer, the analytical engine. She was the first to recognise the machine had applications beyond pure calculation. Lovelace is often considered the first comput


## Build the Chat Bot with LLM

The notebook simulates a chat bot by using TF-IDF retrieval from the Wikipedia page and then selecting relevant sentences. This is a lightweight stand-in for a full LLM-based retrieval system when a hosted model is not directly available.

In [18]:
def get_relevant_chunks(question, top_k=3):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(scores[i], chunks[i]) for i in top_indices]


def answer_question(question):
    relevant = get_relevant_chunks(question)
    extracted_sentences = []
    for score, chunk in relevant:
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', chunk) if s.strip()]
        for sentence in sentences:
            if len(extracted_sentences) >= 4:
                break
            if any(term.lower() in sentence.lower() for term in question.split() if len(term) > 3):
                extracted_sentences.append(sentence)
        if len(extracted_sentences) >= 4:
            break

    if not extracted_sentences:
        extracted_sentences = [re.split(r'(?<=[.!?])\s+', relevant[0][1])[0]]

    return " ".join(extracted_sentences)

print("Chat bot engine ready.")

Chat bot engine ready.


## Ask First Question to Bot

The first question is designed to test whether the bot can identify Ada Lovelace's most important contributions to computing.

In [19]:
question_1 = "What are Ada Lovelace's major contributions to computing?"
answer_1 = answer_question(question_1)
print("Question 1:\n", question_1)
print("\nBot Answer 1:\n", answer_1)

Question 1:
 What are Ada Lovelace's major contributions to computing?

Bot Answer 1:
 On 27 July 2018, Senator Ron Wyden submitted, in the United States Senate, the designation of 9 October 2018 as National Ada Lovelace Day: "To honor the life and contributions of Ada Lovelace as a leading woman in science and mathematics". In his book, Idea Makers, Stephen Wolfram defends Lovelace's contributions. While acknowledging that Babbage wrote several unpublished algorithms for the Analytical Engine prior to Lovelace's notes, Wolfram argues that "there's nothing as sophisticated—or as clean—as Ada's computation of the Bernoulli numbers. Babbage certainly helped and commented on Ada's work, but she was definitely the driver of it." Wolfram then suggests that Lovelace's main achievement was to distill from Babbage's correspondence "a clear exposition of the abstract operation of the machine—something which Babbage never did".


## Ask Second Question to Bot

This follow-up question checks whether the bot can understand the relationship between Ada Lovelace and Charles Babbage and how it shaped her work.

In [20]:
question_2 = "How did her relationship with Charles Babbage influence her work?"
answer_2 = answer_question(question_2)
print("Question 2:\n", question_2)
print("\nBot Answer 2:\n", answer_2)

Question 2:
 How did her relationship with Charles Babbage influence her work?

Bot Answer 2:
 Lovelace's educational and social exploits brought her into contact with scientists such as Andrew Crosse, Charles Babbage, David Brewster, Charles Wheatstone and Michael Faraday, and the author Charles Dickens, contacts which she used to further her education. When she was eighteen, Lovelace's mathematical talents led her to a long working relationship and friendship with fellow British mathematician Charles Babbage. She was particularly interested in Babbage's work on the analytical engine. e and her mother attended one of Charles Babbage's Saturday night soirées with their mutual friend, and Lovelace's private tutor, Mary Somerville.


## Analyze and Evaluate Results

The answer quality is driven by retrieval from the Ada Lovelace article. This approach is good for factual, article-based questions because it directly uses sentences from the source. In my own words, the bot performs well when the question matches explicit phrasing in the page, but it can struggle to infer deeper meaning beyond the text that is present.

Key evaluation points:
- It is strong at locating relevant passages for direct historical questions.
- It is limited by the scope of the text chunks and by not using a true generative model.
- The answers are coherent when the page contains matching facts, but they can become repetitive if the same sentences come from different chunks.
- Future improvements would include a real LLM layer or more advanced summarization of the retrieved passages.